# [방법론 2 · 2026-10-26] 라우터 붕괴 사전 진단 — 4시간·1일 주기

사전 등록(실행 전 고정): D1 분리도 = 평균_t std_e R[t,e] / std_t mean_e R[t,·] (학습 구간 블록 기준), D2 예측가치 = 검증 구간에서
'학습 구간에 맞춘 로지스틱 회귀(라우터 입력 → 다음 블록 최고 전문가)'를 따른 블록 평균 수익 − '학습 최고 고정 전문가' 블록 평균 수익.
라우터 입력은 원본 라우터의 가치 이력 입력을 '학습 최고 고정 전문가의 최근 수익'으로 근사한다(라우터 자기 경로 의존성 제거).
가설: D1 낮음 → 붕괴, D1 높음·D2≤0 → 붕괴 안 하지만 노이즈 전환으로 손해, D1 높음·D2>0 → 쓸모 있는 라우팅.
비교용 실측(이미 확인): 30분 N=1·5·20 전 시드 붕괴, N=10 시드 1개 부분 전환, N=44 거의 붕괴 / 4h·1d 붕괴 / 팩터 전문가(로컬) 붕괴 안 함, D1=0.49.
**학습 없음** — 저장된 모델만 불러온다. CPU로 충분.

- **`런타임 > 모두 실행`만 누르면 된다.** 약 20~40분. `crypto_horizon_retrain_4h/1d`가 끝나 있어야 한다(모델 로드만 함).
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/router_diagnostic/router_diag_hz.csv`

In [ ]:
HORIZON = '4h'   # 아래 루프에서 4h → 1d 순서로 바꿔 가며 실행한다
BUCKET_OF = {'4h': 4 * 3600, '1d': 24 * 3600}
HZ_SEC = BUCKET_OF[HORIZON]; HZ_BARS = HZ_SEC // 300; DAY_BUCKETS = 86400 // HZ_SEC


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = f'hz_{HORIZON}'   # 주기별 별도 폴더 — 기존 rt1006 모델·결과를 건드리지 않는다
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


def separation(R: np.ndarray) -> float:
    common = R.mean(axis=1)
    return float(R.std(axis=1).mean() / (common.std() + 1e-12))


def predictive_value(R: np.ndarray, X: np.ndarray, tr: np.ndarray, va: np.ndarray, seed: int = 0) -> dict:
    y = R.argmax(axis=1)
    best_fixed = int(R[tr].mean(axis=0).argmax())
    out = {"최고 고정 전문가": best_fixed, "검증 최고 고정 수익/블록": float(R[va, best_fixed].mean())}
    classes = np.unique(y[tr])
    if len(classes) < 2:
        pred = np.full(len(va), classes[0])
    else:
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=2000, C=1.0, random_state=seed).fit(sc.transform(X[tr]), y[tr])
        pred = clf.predict(sc.transform(X[va]))
    majority = np.bincount(y[tr], minlength=R.shape[1]).argmax()
    out.update({
        "분류 정확도": float((pred == y[va]).mean()),
        "최빈 기준 정확도": float((y[va] == majority).mean()),
        "분류기 수익/블록": float(R[va, pred].mean()) if len(va) else np.nan,
        "D2 예측가치": float(R[va, pred].mean() - R[va, best_fixed].mean()),
        "분류기 최빈 선택 비율": float(np.bincount(pred, minlength=R.shape[1]).max() / len(pred)),
    })
    return out


def diagnose(R, X, tr, va) -> dict:
    return {"D1 분리도": separation(R[tr]), **predictive_value(R, X, tr, va)}


def expert_step_returns(pool_list, kw, buckets):
    """라우터 학습 때와 같은 방식(스무딩 없이 목표 비중 그대로)으로 각 전문가를 단독 실행해 스텝별 수익을 얻는다."""
    out = []
    for m in pool_list:
        env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **kw)
        obs, _ = env.reset(); done = False
        while not done:
            a, _ = m.predict(obs, deterministic=True)
            obs, _, done, _, _ = env.step(a)
        v = np.asarray(env.value_history, dtype=np.float64)
        out.append(np.diff(v) / v[:-1])
    n = min(len(r) for r in out)
    return np.stack([r[:n] for r in out], axis=1)          # (스텝, 전문가)


def block_data(step_rets, route, macro, best):
    """블록(라우팅 단위) 수익 R과, 블록 시작 시점의 라우터 입력 X(학습 최고 고정 전문가의 최근 macro스텝 수익 — 가치 이력 입력의 근사)."""
    starts = list(range(0, step_rets.shape[0] - route + 1, route))
    R = np.stack([np.prod(1 + step_rets[s:s + route], axis=0) - 1 for s in starts])
    hist = step_rets[:, best]
    X = np.stack([np.concatenate([np.zeros(max(0, macro - s)), hist[max(0, s - macro):s]]) for s in starts])
    return R, X


def diagnose_pool(name, pool_list, kw, fit_buckets, valid_buckets_, route, macro):
    r_fit = expert_step_returns(pool_list, kw, fit_buckets)
    r_val = expert_step_returns(pool_list, kw, valid_buckets_)
    Rf, _ = block_data(r_fit, route, macro, 0)
    best = int(Rf.mean(axis=0).argmax())
    Rf, Xf = block_data(r_fit, route, macro, best)
    Rv, Xv = block_data(r_val, route, macro, best)
    R = np.vstack([Rf, Rv]); X = np.vstack([Xf, Xv])
    tr, va = np.arange(len(Rf)), np.arange(len(Rf), len(R))
    corr = np.corrcoef(Rf.T); pair_corr = float(corr[np.triu_indices_from(corr, 1)].mean())
    d = {'설정': name, **diagnose(R, X, tr, va), '전문가 블록수익 평균 상관': pair_corr, '학습 블록 수': len(Rf)}
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in d.items()})
    return d


rows = []
for HORIZON in ['4h', '1d']:
    HZ_SEC = BUCKET_OF[HORIZON]; HZ_BARS = HZ_SEC // 300; DAY_BUCKETS = 86400 // HZ_SEC
    TAG = f'hz_{HORIZON}'
    run_cells('crypto/notebooks/crypto_hrl_earnhft_colab.ipynb', [2, 4, 6, 8, 9, 44], patch_after=2, subs={2: [('BUCKET_SEC = 1800     # 30분 버킷', 'BUCKET_SEC = HZ_SEC   # [재설계] 주기별 버킷'), ("RL_TRAIN_START = '2025-01-01'", "RL_TRAIN_START = '2021-01-01'"), ("RL_TRAIN_END   = '2025-12-31'", "RL_TRAIN_END   = '2025-09-30'   # [재설계] 검증(10~12월)을 학습에서 제외"), ("RL_EMB_H5      = CRYPTO_DIR / 'crypto_rl_embeddings.h5'", "RL_EMB_H5      = CRYPTO_DIR / f'crypto_rl_embeddings_{HORIZON}.h5'"), ('ROUTE_BUCKETS         = 48 ', 'ROUTE_BUCKETS         = DAY_BUCKETS '), ('MACRO_WINDOW           = 96 ', 'MACRO_WINDOW           = max(2 * DAY_BUCKETS, 14) ')], 6: [("df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])", "df = pd.read_parquet(fp, columns=['datetime', 'close'])\n    df['RetTarget_6b'] = df['close'].shift(-HZ_BARS) / df['close'] - 1   # [재설계] 다음 버킷까지의 수익률(5분 격자는 빈틈 없음)")]})
    pool_list = [PPO.load(str(MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{b}.zip')) for b in POOL_BETAS]
    kw = dict(symbols=symbols, emb_index=emb_index, embs_all=embs_all, ret_index=ret_index, emb_dim=EMB_DIM, tc=TC)
    valid_b = [b for b in all_buckets if '2025-10-01' <= bucket_to_date(b) <= '2025-12-31']
    rows.append(diagnose_pool(f'{HORIZON} 주기', pool_list, kw, train_buckets, valid_b, ROUTE_BUCKETS, MACRO_WINDOW))
DIAG_DIR = DATA_DIR / 'crypto' / 'router_diagnostic'; DIAG_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_csv(DIAG_DIR / 'router_diag_hz.csv', index=False, encoding='utf-8-sig')
print('저장 완료: router_diag_hz.csv')
